# realworld2.ipynb

This notebook was somewhat inspired by a Medium article I recently read which suggests using SQL instead of pandas for increased efficiency.

I will try to rewrite parts of realworld.ipynb to try out this concept.

I already figured out how to store numpy arrays in SQLite without converting to text, by using WKT for example. See `np2sqlite.py`.

In [1]:
from ultralytics.models.sam import SAM3SemanticPredictor
from np2sqlite import array2blob, blob2array
from roadside import test_build_db, get_config
import numpy as np
from icecream import ic
import os
import sqlite3
import cv2
import gc
import torch

from pyefd import elliptic_fourier_descriptors, reconstruct_contour
from shapely.wkt import loads

import pandas as pd
import exif

roadside


# Functions

In [2]:
def get_data_for_detections_table(results_cpu, image_id:int)->pd.DataFrame:
    """ 
    Gets data for for a single image for insertion as records in the 'detections' database table. 
    Returns data as a pandas dataframe containing columns for image_id, class_id, poly_wkt, x_min, y_min, x_max, y_max, confidence
    """

    result = results_cpu[0]
    # Process detection results (assuming one image for simplicity: results,cpu[0])
    image_height = result.orig_shape[0]
    image_width = result.orig_shape[1]

    # create a pandas dataframe for bounding boxes
    boxes_data = result.boxes.data.tolist()
    df_boxes = pd.DataFrame(boxes_data, columns=['x_min', 'y_min', 'x_max', 'y_max', 'confidence', 'class_id'])

    # create a pandas dataframe for segmentation masks (polygons)
    masks_data = []
    # Iterate over each detected object's mask
    for i, mask in enumerate(result.masks.xy):
        poly_arr = mask
        tree_wkt = conv_poly_from_array_to_wkt(poly_arr)
        crown_wkt = get_crown_wkt(image_height, image_width, tree_wkt)
 
        masks_data.append({
            # 'image_path': image_path,
            # 'object_index': i, 
            'class_id': df_boxes.iloc[i]['class_id'], 
            'tree_wkt': tree_wkt,
            'crown_wkt': crown_wkt,
        })
    df_masks = pd.DataFrame(masks_data)  

    # merge df_masks and df_detections  
    df_detections = pd.merge(df_masks, df_boxes, how="outer", left_index=True, right_index=True)
    
    # clean database
    df_detections['image_id'] = image_id
    df_detections.rename(columns={'class_id_x': 'class_id'}, inplace=True)
    df_detections.drop(['class_id_y'], inplace=True, axis='columns')
    df_detections = df_detections.astype({'class_id': int, 'x_min': int, 'y_min': int, 'x_max': int, 'y_max': int})

    return df_detections

# Usage example:

# image_path = 'example_images/08hs-palms-03-zglw-superJumbo.webp'
# text_prompts = ["coconut palm tree"]
# results_gpu = run_sam3_semantic_predictor(input_image_path=image_path, text_prompts=text_prompts)
# results_cpu = [r.cpu() for r in results_gpu] # copy results to CPU
# # delete_results_from_gpu_memory() # Clear GPU memory after processing each image
# get_data_for_images_table(results_cpu)
# # fake_image_id = 999
# get_data_for_detections_table(results_cpu, image_id=fake_image_id)    


In [3]:
def get_data_for_images_table(results_cpu, image_path: str) -> pd.DataFrame:
    """ 
    Gets data for for a single image for insertion as a record in the 'images' database table. 
    Returns a Pandas dataframe containing image_path, image_width, image_height, timestamp, latitude, longitude)
    image_width and image_height come from results_cpu
    timestamp, latitude, longitude come from the EXIF metadata embedded in the image, if it exists. 
    """

    image_height = results_cpu[0].orig_shape[0]
    image_width = results_cpu[0].orig_shape[1]

    with open(image_path, 'rb') as f:
        imgx = exif.Image(f)

    if imgx.has_exif:
        # to see all available exif_data use imgx.get_all()
        
        # timestamp
        timestamp = imgx.datetime
            
        # latitude
        d, m, s = imgx.gps_latitude
        latitude = d + m/60 + s/3600   
        if imgx.gps_latitude_ref == 'S':
            latitude = -latitude  

        # longitude
        d, m, s = imgx.gps_longitude
        longitude = d + m/60 + s/3600   
        if imgx.gps_longitude_ref == 'W':
            longitude = -longitude
        longitude
    else:
        timestamp = pd.NA
        latitude = pd.NA
        longitude= pd.NA
        
    df = pd.DataFrame({
        'image_path': image_path,
        'image_width': image_width,
        'image_height': image_height,
        'timestamp': timestamp,
        'latitude': latitude,
        'longitude': longitude
    },index=[0])
    
    return df

## Usage example:
#
# image_path = image_paths[0]
# results_gpu = rs.run_sam3_semantic_predictor(input_image_path=image_path, text_prompts=text_prompts)
# results_cpu = [r.cpu() for r in results_gpu] # copy results to CPU
# _results_from_gpu_memory() # Clear GPU memory after processing each image
# get_data_for_images_table(results_cpu)


In [4]:
def run_sam3_semantic_predictor(input_image_path: str, text_prompts: list=['coconut palm tree']) -> list:
    """ 
    Uses the SAM3 semantic predictor to detect objects specified by text prompts in an image.
    
    Inputs:
      input_image_path relative to working directory 
      text_prompts: list of text prompts; default: ['coconut palm tree']
      
    Outputs:
      results:     
    """
    # Initialize predictor with configuration
    overrides = dict(
        conf=0.25,
        task="segment",
        mode="predict",
        model="sam3.pt",
        half=True,  # Use FP16 for faster inference
        save=True,  # Save image visualizing output results
        save_txt=False,  # Save output results in text format
        save_conf=False,  # Save confidence scores   
        imgsz=1932,  # Adjusted image size from 1920 to meet stride 14 requirement
        batch=1,
        device="0",  # Use GPU device 0
    )
    predictor = SAM3SemanticPredictor(overrides=overrides)

    # Set image once for multiple queries
    predictor.set_image(input_image_path)

    # Query with multiple text prompts
    results = predictor(text=text_prompts)

    return results

## Example usage:

# root_dir = "/home/aubrey/Desktop/sam3-2026-01-31"
# image_paths = ["20251129_152106.jpg", "08hs-palms-03-zglw-superJumbo.webp"]
# text_prompts = ["coconut palm tree"]

# os.chdir(root_dir) # ensure we start in the correct directory
# for image_path in image_paths:
#     results_gpu = run_sam3_semantic_predictor(image_path, text_prompts)

#     # Free up GPU memory in preparation for detecting objects in the next image
#     # This is a work-around to prevent out-of-memory errors from the GPU
#     # I move all results for further processing and use the GPU only for object detection.
#     print('deleting results from GPU memory')       
#     results_cpu = [r.cpu() for r in results_gpu] # copy results to CPU
#     delete_results_from_gpu_memory()

# print("Processing complete.")


In [ ]:
def build_db(db_path, image_paths, schema_sql) -> None:
    # for testing:
    os.remove(db_path) if os.path.exists(db_path) else None

    # If it does not exist, create the SQLite database and tables based on the provided schema SQL; 
    if not os.path.exists(db_path): 
        print(f"Database '{db_path}' does not exist. Creating new database and tables.") 
        conn = sqlite3.connect(db_path)
        conn.executescript(schema_sql)
        conn.commit()
        # conn.close()
        
    for image_path in image_paths:
        # ensure GPU memory is empty before processing image
        # del results_gpu
        if "model" in globals():
            print('deleting model from GPU memory')
            del model
        gc.collect()
        torch.cuda.empty_cache()
            
        # run the SAM3 semantic predictor on an image and move results to CPU for further processing
        results_gpu = run_sam3_semantic_predictor(
            input_image_path=image_path, 
            text_prompts=["coconut palm tree"]
        )
        
        # Free up GPU memory in preparation for detecting objects in the next image
        # This is a work-around to prevent out-of-memory errors from the GPU
        # I move all results for further processing and use the GPU only for object detection.
        print('copying results_gpu to results_cpu')
        results_cpu = [r.cpu() for r in results_gpu] # copy results to CPU
        print('deleting results_gpu from GPU')       
        del results_gpu 
        gc.collect() 
        torch.cuda.empty_cache() # Clears unoccupied cached memory
        
        conn = sqlite3.Connection(db_path)
        
        # add record to images table
        #############################
        image_height = results_cpu[0].orig_shape[0]
        image_width = results_cpu[0].orig_shape[1]
        image_cursor = conn.execute(
            "INSERT INTO images (image_path, image_width, image_height) VALUES (?, ?, ?)", 
            (image_path, image_width, image_height)
        )
        image_id = image_cursor.lastrowid
        conn.commit()
        
        with open(image_path, 'rb') as f:
            imgx = exif.Image(f)
            if imgx.has_exif:
                # timestamp
                timestamp = imgx.datetime
                    
                # latitude
                d, m, s = imgx.gps_latitude
                latitude = d + m/60 + s/3600   
                if imgx.gps_latitude_ref == 'S':
                    latitude = -latitude              

                # longitude
                d, m, s = imgx.gps_longitude
                longitude = d + m/60 + s/3600   
                if imgx.gps_longitude_ref == 'W':
                    longitude = -longitude
                longitude
                
                conn.execute(
                    "UPDATE images SET timestamp = ?, latitude = ?, longitude = ? WHERE image_path = ?",
                    (timestamp, latitude, longitude, image_path)
                )
                conn.commit()
                
        # add records to detections table
        #################################
        
        cpu_results = results_cpu # FIX THIS
        
        boxes = cpu_results[0].boxes
        conf_list = boxes.conf.cpu().numpy().tolist()
        class_list = boxes.cls.cpu().numpy().tolist()
        tree_contour_list = cpu_results[0].masks.xy

        for i, tree_contour in enumerate(tree_contour_list):
            # 1. Ensure it's a mutable NumPy array or list
            # YOLO .xy returns a float32 numpy array
            if len(tree_contour) == 0:
                continue

            # 2. Check if the last coordinate matches the first
            # tree_contour[0] is first point [x, y], tree_contour[-1] is last point [x, y]
            if not np.array_equal(tree_contour[0], tree_contour[-1]):
                # Append the first point to the end to close the loop
                tree_contour = np.vstack([tree_contour, tree_contour[0]])

            # 4. Insert into the database
            
            class_id = class_list[i]
            confidence = conf_list[i]
            conn.execute(
                "INSERT INTO detections (image_id, class_id, tree_contour, confidence) VALUES (?, ?, ?, ?)", 
                (image_id, class_id, array2blob(tree_contour), confidence)
            )

        # Commit changes and close the connection
        conn.commit()
        conn.close()
        
# usage example:
build_db(
    db_path = 'newt.db', 
    image_paths = [
        'data_cache/example_images/20251129_152106.jpg',
        'data_cache/example_images/data_cache/example_images/08hs-palms-03-zglw-superJumbo.webp',
        ], 
    schema_sql = config['default_schema_sql']
    )

Database 'newt.db' does not exist. Creating new database and tables.
Ultralytics 8.4.49 🚀 Python-3.13.11 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3080 Laptop GPU, 15982MiB)

image 1/1 /home/aubrey/Desktop/crb-2026-05-13/data_cache/example_images/20251129_152106.jpg: 1932x1932 26 coconut palm trees, 1299.6ms
Speed: 13.0ms preprocess, 1299.6ms inference, 3.9ms postprocess per image at shape (1, 3, 1932, 1932)
Results saved to /home/aubrey/Desktop/blog2026/runs/segment/predict-64
copying results_gpu to results_cpu
deleting results_gpu from GPU


ic| image_id: 1
ic| class_id: 0.0
ic| confidence: 0.91748046875
ic| image_id: 1
ic| class_id: 0.0
ic| confidence: 0.84619140625
ic| image_id: 1
ic| class_id: 0.0
ic| confidence: 0.83544921875
ic| image_id: 1
ic| class_id: 0.0
ic| confidence: 0.8115234375
ic| image_id: 1
ic| class_id: 0.0
ic| confidence: 0.8017578125
ic| image_id: 1
ic| class_id: 0.0
ic| confidence: 0.74365234375
ic| image_id: 1
ic| class_id: 0.0
ic| confidence: 0.74169921875
ic| image_id: 1
ic| class_id: 0.0
ic| confidence: 0.72314453125
ic| image_id: 1
ic| class_id: 0.0
ic| confidence: 0.7080078125
ic| image_id: 1
ic| class_id: 0.0
ic| confidence: 0.69287109375
ic| image_id: 1
ic| class_id: 0.0
ic| confidence: 0.6728515625
ic| image_id: 1
ic| class_id: 0.0
ic| confidence: 0.6328125
ic| image_id: 1
ic| class_id: 0.0
ic| confidence: 0.56640625
ic| image_id: 1
ic| class_id: 0.0
ic| confidence: 0.55322265625
ic| image_id: 1
ic| class_id: 0.0
ic| confidence: 0.53515625
ic| image_id: 1
ic| class_id: 0.0
ic| confidence: 0.52

Ultralytics 8.4.49 🚀 Python-3.13.11 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3080 Laptop GPU, 15982MiB)

image 1/1 /home/aubrey/Desktop/crb-2026-05-13/data_cache/example_images/data_cache/example_images/08hs-palms-03-zglw-superJumbo.webp: 1932x1932 2 coconut palm trees, 1281.6ms
Speed: 14.5ms preprocess, 1281.6ms inference, 1.6ms postprocess per image at shape (1, 3, 1932, 1932)
Results saved to /home/aubrey/Desktop/blog2026/runs/segment/predict-65
copying results_gpu to results_cpu
deleting results_gpu from GPU


ic| image_id: 2
ic| class_id: 0.0
ic| confidence: 0.94775390625
ic| image_id: 2
ic| class_id: 0.0
ic| confidence: 0.93994140625


In [6]:
def reconstruct_aligned_mask(image_shape, contour, order=10, align_to_centroid=True):
    """
    Finds EFDs and reconstructs the mask perfectly aligned with the original locus.
    
    Parameters:
        image_shape (tuple): Shape of the original image (H, W)
        contour (ndarray): Contour array of original image; shape (N, 2) or (N, 1, 2)
        order (int): Number of Fourier coefficients to use
        
    Returns:
        ndarray: Binary mask with the reconstructed shape in the correct position
    """
    
    ic()
    
    # 1. Standardize contour shape to (N, 2)
    contour = contour.reshape(-1, 2)
    
    # 2. Calculate the true centroid (locus) of the original contour using moments.
    # This keeps the reconstructed shape strictly bound to the true defect location.
    M = cv2.moments(contour)
    if M["m00"] != 0:
        cX = M["m10"] / M["m00"]
        cY = M["m01"] / M["m00"]
    else:
        cX, cY = np.mean(contour, axis=0)

    # 3. Compute EFD coefficients (keeping unnormalized to retain spatial properties)
    coeffs = elliptic_fourier_descriptors(contour, order=order, normalize=False)
    
    # 4. Corrected function: Reconstruct contour points via the native API.
    # We pass the calculated cX, cY into the locus argument.
    # Next line added by Aubrey Moore 2026-06-02
    num_points = contour.shape[0]  # Use the original number of contour points for reconstruction
    reconstructed_points = reconstruct_contour(coeffs, locus=(cX, cY), num_points=num_points)
    
    # 5. Prevent sub-pixel "floor bias" shift by rounding before converting to integer
    reconstructed_contour = np.round(reconstructed_points).astype(np.int32)
    reconstructed_contour = reconstructed_contour.reshape(-1, 1, 2)
    
    # 6. Create the aligned mask
    reconstructed_mask = np.zeros(image_shape, dtype=np.uint8)
    cv2.drawContours(reconstructed_mask, [reconstructed_contour], -1, 255, -1)
    
    if align_to_centroid:
        # Calculate the centroid of the reconstructed mask
        M_recon = cv2.moments(reconstructed_contour)
        if M_recon["m00"] != 0:
            recon_cX = M_recon["m10"] / M_recon["m00"]
            recon_cY = M_recon["m01"] / M_recon["m00"]
        else:
            recon_cX, recon_cY = np.mean(reconstructed_contour.reshape(-1, 2), axis=0)
        
        # Calculate the shift needed to align the reconstructed contour's centroid with the original
        shift_x = int(cX - recon_cX)
        shift_y = int(cY - recon_cY)
        ic(shift_x, shift_y)
        
        # Shift the reconstructed contour and mask
        translation_matrix = np.float32([[1, 0, shift_x], [0, 1, shift_y]])
        reconstructed_mask = cv2.warpAffine(reconstructed_mask, translation_matrix, (image_shape[1], image_shape[0]))
        reconstructed_contour = cv2.transform(reconstructed_contour, translation_matrix)
    
    return reconstructed_contour, reconstructed_mask


In [7]:
def get_centroid(contour):
    """ Returns centroid of a contour. """
    M = cv2.moments(contour)
    if M["m00"] != 0:
        cX = int(M["m10"] / M["m00"])
        cY = int(M["m01"] / M["m00"])
    else:
        cX, cY = np.mean(contour, axis=0)
    return cX, cY    

In [8]:
def calc_defect_contours(image_height, image_width, tree_wkt, order, minpixels):  
    poly = loads(tree_wkt)
    coords = list(poly.exterior.coords)    
    tree_contour = np.array(coords, dtype=np.int32).reshape(-1, 1, 2)
    
    canvas = np.zeros((image_height, image_width), np.uint8)
    tree_mask = cv2.drawContours(canvas, [tree_contour], -1, 255, -1)
    tree_mask_cx, tree_mask_cy = get_centroid(tree_mask)
    
    _, reconstructed_mask = reconstruct_aligned_mask(image_shape=(image_height, image_width), contour=tree_contour, order=order)
    # reconstructed_mask_cx, reconstructed_mask_cy = get_centroid(reconstructed_mask)
    registered_mask = reconstructed_mask.copy()
    additions_mask = cv2.bitwise_and(registered_mask, cv2.bitwise_not(tree_mask))    
    defect_contours, _ = cv2.findContours(additions_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    defect_contours = [cnt for cnt in defect_contours if cv2.contourArea(cnt) > minpixels]

    return defect_contours

# MAIN

In [9]:
config = get_config()
for key, value in config.items():
    ic(key, value)

SHA256 hash of downloaded file: bb2223e68e5eabcb85fef2a89237cdb99b9f613151bcecb5bdba7b663ddbfedb
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.
ic| key: 'default_schema_sql'
    value: '''    CREATE TABLE IF NOT EXISTS images (
                    image_id INTEGER PRIMARY KEY AUTOINCREMENT,
                    image_path TEXT UNIQUE,
                    image_width INTEGER,
                    image_height INTEGER,
                    timestamp TEXT,
                    latitude REAL,
                    longitude REAL
                );
                CREATE TABLE IF NOT EXISTS detections (
                    detection_id INTEGER PRIMARY KEY AUTOINCREMENT,
                    image_id INTEGER,
                    class_id INTEGER,
                    tree_contour BLOB,
                    confidence REAL,
                    FOREIGN KEY (image_id) REFERENCES images (image_id) ON DELETE C

In [10]:
if not os.path.exists(config['dbpath']):
    test_build_db()

conn = sqlite3.connect(config['dbpath'])
# conn.row_factory = sqlite3.Row # allows us to access columns by name

# FOR TESTING, WE WILL REMOVE ALL DATA FROM THE DAMAGE TABLE TO START FRESH
conn.execute('DELETE FROM damage')
conn.commit()

sql = 'select image_id, image_path, image_width, image_height from images'
for image_row in conn.execute(sql):
    image_id, image_path, image_width, image_height = image_row
    
    sql = 'SELECT detection_id, tree_wkt FROM detections WHERE image_id=?'
    for detection_row in conn.execute(sql, (image_id,)):
        detection_id, tree_wkt = detection_row
        ic('processing image', image_id, detection_id)
        
        defect_contours = calc_defect_contours(image_height, image_width, tree_wkt, config['order'], config['minpixels'])
        ic(len(defect_contours))       
        for cnt in defect_contours:
            cnt_blob = array2blob(cnt)
            conn.execute('INSERT INTO damage (detection_id, defect_contour) VALUES (?, ?)', (detection_id, cnt_blob))
conn.commit()

conn.close()

ic('FINISHED');

ic| 'processing image', image_id: 1, detection_id: 1
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 19:37:16.084
ic| shift_x: 3, shift_y: -41
ic| len(defect_contours): 14
ic| 'processing image', image_id: 1, detection_id: 2
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 19:37:16.117
ic| shift_x: 0, shift_y: 41
ic| len(defect_contours): 20
ic| 'processing image', image_id: 2, detection_id: 3
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 19:37:16.141
ic| shift_x: 10, shift_y: 85
ic| len(defect_contours): 13
ic| 'processing image', image_id: 2, detection_id: 4
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 19:37:16.159
ic| shift_x: 3, shift_y: 5
ic| len(defect_contours): 2
ic| 'processing image', image_id: 2, detection_id: 5
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 19:37:16.176
ic| shift_x: -3, shift_y: 5
ic| len(defect_contours): 0
ic| 'processing image', image_id: 2, detection_id: 6
ic| 3337628227.py:14 in reconstruct_aligned_mask() at 19:37:16.